#### import 및 기본변수

In [11]:
import pandas as pd
import numpy as np
import re, os, functions
from os.path import join
import warnings
warnings.simplefilter("ignore")

debt_dtype = {'채무자키':str, '타채무자키':str, '담당자키':str, '관리자비고':str}
account_dtype = {'채무자키':str, '계좌키':str, '타채무자키':str}
grt_dtype = {'채무자키':str, '계좌키':str, '타채무자키':str, '보증인키':str}
rehabilitation_dtype = {'채무자키':str, '계좌키':str, '분납키':str, '사건키':str, '신고계좌':str, '입금계좌':str}
credit_dtype = {"채무자키":str, "계좌키":str, '보증인키':str, '신용회복키' : str, "계좌번호":str, "심의차수":str, "변제금수취계좌":str}
event_dtype = {'채무자키':str, '법조치키':str, '계좌키':str, '관련법조치키':str, '법취하키':str, '타법조치키':str, '타채무자키':str, '관할법원코드':str}
deposit_dtype = {'채무자키':str, '입금키':str, '계좌키':str, '계좌번호':str, '입금고정키':str, '타채무자키':str}
reduction_dtype = {'채무자키':str, '계좌키':str, '감면키':str}
installment_dtype = {'채무자키':str, '계좌키':str, '분납키':str}

##################################
company = "대성"      # 솔림 or 대성
basedate = "260331" 
previous_cutoff = "260228" # 계좌 
wd = join(r"D:\3.자산\전산 dataset", company)
##################################
ext = ".xlsx"  
read_dir = join(wd, basedate)
write_dir = join(wd, basedate, "parquet")
previous_read_dir = join(wd, previous_cutoff, "parquet")

# parquet 폴더 만들기
if not os.path.exists(write_dir) :
    os.makedirs(write_dir)


# 작업전 할일 ################################
"""
모든 데이터 다운받기 전에
아래 내용 사전점검 할 것! 이것만 해둬도 파일 다운 받고 기준데이터 만들어도 될듯?
"""
# print('신복 특별면책, 반송 확인 후 전산 바로 수정!') : 진행상황에 반영되므로 후수정 해도 상관없음. 다만 새채무상태 작성시 유의
print('개회 다운 전, 0.20526551(통합신고건)에 입금 잡혔는지 확인, 1.현재결과 항고건 확인, 2.인가 but 변제조회실패 3.당월 중복입금(pk:계좌키&입금일&입금합계')
print('개회 엑셀 받은 후 최근 면책건 채무/보증인 상태값만 바로 수정(담당자x)')
print('채무자 다운전 종결요청건 처리. 다운 후 법인의 채무상태 확인 및 수정(법인차주가 실제 약정건인지만 확인(a), 보증상태값을 채무상태에 작성한경우(b)는 수정불요. 그외 상태값은 확인후 수정)')
print('    현재 a : 성은개발, b : 거성산업, 동진산업, 관들')
print('계좌 다운 후 종결일 없는데 담당자 종결, 매각, 환매, ds인 건 확인 및 수정')
print('법조치 이달 등록건 중 파산, 상속건 구분,비고,사건번호 확인 / 최신화 안 된 것, 대상자 여러명인건 확인')
print('감면요청 건 중 입금되었는데 승인 안 난 건')
print('입금 : 당월 및 후일입금 다운로드')

# 1) object 컬럼 → str
# for col in df.select_dtypes(include='object').columns:
#     df[col] = df[col].astype(str)

개회 다운 전, 0.20526551(통합신고건)에 입금 잡혔는지 확인, 1.현재결과 항고건 확인, 2.인가 but 변제조회실패 3.당월 중복입금(pk:계좌키&입금일&입금합계
개회 엑셀 받은 후 최근 면책건 채무/보증인 상태값만 바로 수정(담당자x)
채무자 다운전 종결요청건 처리. 다운 후 법인의 채무상태 확인 및 수정(법인차주가 실제 약정건인지만 확인(a), 보증상태값을 채무상태에 작성한경우(b)는 수정불요. 그외 상태값은 확인후 수정)
    현재 a : 성은개발, b : 거성산업, 동진산업, 관들
계좌 다운 후 종결일 없는데 담당자 종결, 매각, 환매, ds인 건 확인 및 수정
법조치 이달 등록건 중 파산, 상속건 구분,비고,사건번호 확인 / 최신화 안 된 것, 대상자 여러명인건 확인
감면요청 건 중 입금되었는데 승인 안 난 건
입금 : 당월 및 후일입금 다운로드


#### 개인회생

In [15]:
# 엑셀 읽기, 정렬(사건-컷오프-채권번호), parquet 저장
filename = functions.키워드로파일명찾기(read_dir, "개인회생새창", "기준데이터", 전체경로=False).split('.')[0]
df = pd.read_excel(join(read_dir, filename+".xlsx"), dtype=rehabilitation_dtype).fillna("")

# 진행내용 표시 했는지 체크
if df.loc[0,"진행내용"] == "" :
    print("개인회생 엑셀다운로드시 우클릭후 '1.채권자목록,일반내용,진행내용 표시'를 선택해야 합니다.")

# parquet 저장
functions.to_parquet(df, join(write_dir, filename+".parquet"))

#### 전산 raw_data

In [13]:
#########################################
# 초본 > 자택 > 직장 > 기타 우선순위 적용은 항상 한다.
파일검색어 = { # [타입, 제외키워드]
         
    "입금조회.*당월" : [deposit_dtype,""],
    "보증인새창" : [grt_dtype,"새보증인상태"],
    
    "계좌조회" : [account_dtype,""],
    
    "채무자조회" : [debt_dtype,""],
    "법조치조회" : [event_dtype,""],
    
    "감면조회" : [reduction_dtype,""],
    "약정분납새창" : [installment_dtype,"기준데이터"]
}
#########################################

# 이전계좌 (종결확인)
account_previous = pd.read_parquet(functions.키워드로파일명찾기(join(previous_read_dir), 포함키워드="계좌조회새창", 전체경로=True))


# 계좌, 보증인새창 읽을 때면 채무상태 누락건 없는지 미리 확인
if ("계좌조회" in 파일검색어.keys()) or ("보증인새창" in 파일검색어.keys()) :
    # 상태정리파일
    path_상태정리 = r"D:\3.자산\전산 dataset\상태정리.xlsx"
    상태정리 = pd.read_excel(path_상태정리, sheet_name="상태", header=1)
    메모정리 = pd.read_excel(path_상태정리, sheet_name="메모")
    담당자정리 = pd.read_excel(path_상태정리, sheet_name="담당자")

    
for i, v in 파일검색어.items() : 
    print(i, '파일변환시작')
    filename = os.path.splitext(functions.키워드로파일명찾기(read_dir, i, v[1],전체경로=False))[0]
    df = pd.read_excel(join(wd, basedate, filename+ext), dtype = v[0]).fillna("")
    
    # 1) 상태정리에 없는 상태값 확인(있으면 상태정리 파일 업데이트 해야! 수정건 있으면 break)
    # 2) 담당자와 메모가 종결값인데 종결일 없는 경우
    # 3) 보증인 주소 보정
    if re.search("계좌|보증인", i) :
        # 계좌
        if re.search("계좌", i) : 
            상태칼럼명 = "채무상태"
            출력cols = ["채무자키","계좌키", 상태칼럼명, "담당자","종결일"]
            
            # 종결해제건 확인 -----
            전달종결건 = account_previous[account_previous.종결일!=""]["계좌키"]
            종결해제건 = df[(df.종결일=="") & df.계좌키.isin(전달종결건)][출력cols]
            if not 종결해제건.empty :
                functions.display_with_explain(종결해제건, "종결해제건")
            
        # 보증인    
        else : 
            상태칼럼명 = "보증인상태"
            출력cols = ["채무자키","계좌키","보증인키", 상태칼럼명, "담당자","종결일"]
            
            # 보증인 주소 보정
            cols = ["초본주소", "초본번지인", "초본우편번호", "자택주소","자택번지인","자택우편번호","직장주소","직장번지인", "직장우편번호"]
            df[cols] = df[cols].applymap(lambda x: x.strip() if isinstance(x, str) else x)


            # 주소채우기 : 초본이 없는 경우 자택 > 직장을 초본칸에 넣기
                # values를 붙이거나 to_numpy()를 붙여야함.(안그럼 정렬과 인덱스유지 사이의 충돌로 na값이 입력됨)
                # 이제 isna()조건은 없어도 되지만 혹시 몰라서 추가함
            mask = ((df.초본주소=="") | (df.초본주소.isna()))
            df.loc[mask,["초본주소", "초본번지인", "초본우편번호"]] = df.loc[mask, ["자택주소", "자택번지인", "자택우편번호"]].to_numpy()
            mask = ((df.초본주소=="") | (df.초본주소.isna()))
            df.loc[mask,["초본주소", "초본번지인", "초본우편번호"]] = df.loc[mask, ["직장주소", "직장번지인", "직장우편번호"]].to_numpy()
            
        
        # 공통
        미종결조건 = (df["종결일"]=="") # fillna('')했음
        
        # 상태정리에 없는 상태값 확인
        상태수정할건 = df[미종결조건 & (~df[상태칼럼명].str.replace("_","").isin(상태정리.채무상태)|(df[상태칼럼명]==""))]
        if not 상태수정할건.empty : 
            functions.display_with_explain(상태수정할건[출력cols], "상태정리에 없는 상태값 확인")

    
    # 채무자 관리자비고(새채무자키) 검사
    # 채무자 주소채우기
    if re.search("채무자", i) :
        if len(df.query("관리자비고==''")) > 1 :
            print("새채무자키 부여 필요")
            print(len(nothing)) # 오류를 위한 코드
        
        # 예수금제외
        df.drop(df[(df.성명=="예수금")].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        
        # 채무자 주소 보정  
        cols = ["초본주소", "초본번지인", "초본우편번호", "자택주소","자택번지인","자택우편번호","직장주소","직장번지인", "직장우편번호","기타주소", "기타번지인", "기타우편번호"]
        df[cols] = df[cols].applymap(lambda x: x.strip() if isinstance(x, str) else x)


        # 주소채우기 : 초본이 없는 경우 자택 > 직장 > 기타 주소를 초본칸에 넣기
            # values를 붙이거나 to_numpy()를 붙여야함.(안그럼 정렬과 인덱스유지 사이의 충돌로 na값이 입력됨)
            # 이제 isna()조건은 없어도 되지만 혹시 몰라서 추가함
        mask = ((df.초본주소=="") | (df.초본주소.isna()))
        df.loc[mask,["초본주소", "초본번지인", "초본우편번호"]] = df.loc[mask, ["자택주소", "자택번지인", "자택우편번호"]].to_numpy()
        mask = ((df.초본주소=="") | (df.초본주소.isna()))
        df.loc[mask,["초본주소", "초본번지인", "초본우편번호"]] = df.loc[mask, ["직장주소", "직장번지인", "직장우편번호"]].to_numpy()
        mask = ((df.초본주소=="") | (df.초본주소.isna()))
        df.loc[mask,["초본주소", "초본번지인", "초본우편번호"]] = df.loc[mask, ["기타주소", "기타번지인", "기타우편번호"]].to_numpy()
    
    # 계좌 예수금 제거
    if re.search("계좌", i) :
        df.drop(df[(df.채무자명=="예수금")].index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        
    # 법조치 법원사건번호 없는 것 제거
    if re.search('법조치', i) :
        index =  df[(df.관할법원=="0") | (df.관할법원=="") | (df.사건번호=="0") | (df.사건번호=="")].index
        df.drop(index, inplace=True)
        df.reset_index(drop=True, inplace=True)
        # 사건번호 역순정렬
        df.sort_values('사건번호', ascending=False, inplace=True)
    
    # parquet 저장
    functions.to_parquet(df, join(write_dir, filename+".parquet"), engine="pyarrow")

입금조회.*당월 파일변환시작
보증인새창 파일변환시작
계좌조회 파일변환시작
채무자조회 파일변환시작
법조치조회 파일변환시작
감면조회 파일변환시작
약정분납새창 파일변환시작


#### 신용회복 계좌별진행상황

In [14]:
# 계좌별 진행상황 파일 읽기
########################################################
약식병합 = True ############# 금요일(일요일)이 컷오프가 아니라면 약식병합해야-------
basedate = basedate # "250423" # 정식 업로드결과 파일명의 날짜를 컷오프일에 맞춰 수정할 것
company = company  # 솔림 or 대성
########################################################

nauri_dtype = {"채무자키":str, "계좌키":str, '보증인키':str, '신용회복키' : str, "계좌번호":str, "심의차수":str, "변제금수취계좌":str}
path_base = r"D:\3.자산\신용회복\신용회복 전체데이터\계좌별 진행상황"

# parquet 폴더 없으면 만들기
write_path_base = join(path_base, "parquet")
if not os.path.exists(write_path_base) :
    os.mkdir(write_path_base)

# 읽을 파일명
suffix = "통합" if company!="대성" else "대성"

filename = basedate + "_계좌별 진행상황 조회_" + suffix

filename_정식 = filename + "_업로드결과"
filename_약식 = filename + "_약식_업로드결과"

# 계좌별 진행상황 조회 파일 읽기 및 parquet 저장
    # 정식
정식_ori = pd.read_excel(join(path_base, filename_정식+".xlsx" ), dtype=nauri_dtype).fillna("")
functions.to_parquet(정식_ori, join(write_path_base, filename_정식+".parquet"))
    # 약식
if 약식병합 : 
    약식_ori = pd.read_excel(join(path_base, filename_약식+".xlsx"), dtype=nauri_dtype).fillna("")
    functions.to_parquet(약식_ori, join(write_path_base, filename_약식+".parquet"))

# 이건 기준데이터 아니니 파일 옮길 필요 없음.

#### 수정

In [8]:
# 채무자, 계좌 : 채무상태, 담당자 
######################################
수정데이터 = {
    "20411254": {"담당자": "회사보유","채무상태": "파산 누락"},
    # "20451111": {"담당자": "개인회생","채무상태": "개인회생(진행중)"}
}
######################################

# 수정 대상 키
수정할키 = 수정데이터.keys()

# 파일 읽기
path_debt = functions.키워드로파일명찾기(write_dir, "채무자조회", 여러파일허용=False)
path_account = functions.키워드로파일명찾기(write_dir, "계좌조회", 여러파일허용=False)

debt = pd.read_parquet(path_debt)
account = pd.read_parquet(path_account)

# 수정 대상 키가 채무자키에 존재하는지 확인
missing_in_debt = set(수정할키) - set(debt.채무자키)
missing_in_account = set(수정할키) - set(account.채무자키)

if missing_in_debt or missing_in_account:
    raise ValueError(
        f"""
        debt 누락: {missing_in_debt}
        account 누락: {missing_in_account}
        """
    )


# 수정
for df in [debt, account]:
    df["채무자키"] = df["채무자키"].astype(str)

    for col in ["담당자", "채무상태"]:
        df.loc[
            df.채무자키.isin(수정할키),
            col
        ] = df["채무자키"].map(
            {k: v[col] for k, v in 수정데이터.items()}
        )
    
# 파일 저장
functions.to_parquet(debt, path_debt, engine="pyarrow")
functions.to_parquet(account, path_account, engine="pyarrow")

In [9]:
# 계좌종결일 수정 (계좌키!!)
######################################
수정할계좌키 = ["200964644"]
종결일 = ["2023-05-02"]
######################################

# 매핑 딕셔너리 생성
종결일_매핑 = dict(zip(수정할계좌키, 종결일))

# 파일 읽기
path_account = functions.키워드로파일명찾기(write_dir, "계좌조회", 여러파일허용=False)

account = pd.read_parquet(path_account)

# 수정
account.loc[
    account.계좌키.isin(종결일_매핑.keys()),
    "종결일"
] = account["계좌키"].map(종결일_매핑)

# 파일 저장
functions.to_parquet(account, path_account, engine="pyarrow")

In [ ]:
# 보증인
######################################
수정할보증인키 = ["200191293"]
수정할보증인상태 = ["파산(진행확정)_"]
######################################

# 매핑 딕셔너리 생성
보증인상태_매핑 = dict(zip(수정할보증인키, 수정할보증인상태))

# 파일 읽기
path_grt = functions.키워드로파일명찾기(write_dir, "보증인새창", 제외키워드="새보증인", 여러파일허용=False)
grt = pd.read_parquet(path_grt)

# 수정
grt.loc[
    grt.보증인키.isin(보증인상태_매핑.keys()),
    "보증인상태"
] = grt["보증인키"].map(보증인상태_매핑)

# 파일 저장
functions.to_parquet(grt, path_grt, engine="pyarrow")

### 나의 사건

### 기타

#### 단일파일

In [ ]:
#################################
path_read_file = r"D:\3.자산\전산 dataset\대성\250930\통합dc_for_v2\법조치조회새창_20251028_1642.xlsx"
#################################
dtype_to_use = functions.통합_dtype

df = pd.read_excel(join(path_read_file), dtype=dtype_to_use)
functions.to_parquet(df, join(os.path.dirname(path_read_file), os.path.splitext(os.path.basename(path_read_file))[0]+".parquet"), engine="pyarrow")

#### 폴더

In [ ]:
##########################################################
path_read = r"D:\3.자산\전산 dataset\대성\250930\통합dc_for_v2\법조치조회새창_20251028_1642.xlsx"
##########################################################
dtype_to_use = functions.통합_dtype
p_extension = re.compile('xlsx$', re.I)
file_list = [f.name for f in os.scandir(path_read) if f.is_file() and p_extension.search(f.name)]

path_write = join(path_read, "parquet")
if not os.path.exists(path_write) :
    os.mkdir(path_write)

for f in file_list :
    print("변환시작 :",f)
    fn = os.path.splitext(f)[0]
    df = pd.read_excel(join(path_read, f), dtype=dtype_to_use)
    functions.to_parquet(df, join(path_write, fn+".parquet"), engine="pyarrow")